# 🌍 AirSentinel Cameroun
## Notebook 05 — IRS par ACP + Épisodes climatiques
**Responsable : BODEHOU Sabine — ISSEA**

### Objectif
1. Calculer l'IRS par ACP sur les 36 moyennes de villes
2. Définir les seuils par percentiles 50/75/90
3. Valider avec les seuils OMS 2021

### Références
- WHO AQG 2021 — NCBI NBK574591
- Ceccherini et al. 2017 — percentile 90

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('../data/processed/dataset_enrichi.xlsx')
df['date'] = pd.to_datetime(df['date'])
print(f'✅ Dataset chargé : {df.shape[0]:,} lignes')

## Étape 1 — ACP sur les 36 moyennes de villes
**Pourquoi les moyennes ?**
Les 36 villes ont des profils climatiques indépendants.
Les données journalières se ressemblent d'un jour à l'autre (autocorrélation).
→ On prend 1 moyenne par ville = 36 lignes indépendantes ✅

In [ ]:
cols_irs = ['pm2_5_moyen', 'dust_moyen', 'co_moyen', 'uv_moyen', 'ozone_moyen']
cols_irs = [c for c in cols_irs if c in df.columns]

# Moyennes par ville (36 lignes indépendantes)
df_moyennes = df.groupby('ville')[cols_irs].mean().dropna()
print(f'ACP sur {len(df_moyennes)} profils de villes indépendants')

# Normalisation
scaler_acp = StandardScaler()
X_moy      = scaler_acp.fit_transform(df_moyennes)

# ACP
pca_test = PCA()
pca_test.fit(X_moy)

print('\nVariance expliquée par composante :')
cumul = 0
for i, var in enumerate(pca_test.explained_variance_ratio_):
    cumul += var
    print(f'  Axe {i+1} : {var*100:.1f}%  → cumulé : {cumul*100:.1f}%')

In [ ]:
# Choisir le nombre d'axes selon variance expliquée
variance_cumulee = np.cumsum(pca_test.explained_variance_ratio_)
n_axes = np.argmax(variance_cumulee >= 0.80) + 1
print(f'Nombre d axes retenus : {n_axes} (expliquent {variance_cumulee[n_axes-1]*100:.1f}% de la variance)')

# ACP finale
pca = PCA(n_components=n_axes)
pca.fit(X_moy)

print('\nPoids IRS calculés par ACP :')
if n_axes == 1:
    poids = abs(pca.components_[0])
    poids = poids / poids.sum()
    for col, p in zip(cols_irs, poids):
        print(f'  {col:20s} : {p:.4f} ({p*100:.1f}%)')
else:
    v1 = pca.explained_variance_ratio_[0]
    v2 = pca.explained_variance_ratio_[1]
    total = v1 + v2
    print(f'  Axe 1 ({v1*100:.1f}%) : ', dict(zip(cols_irs, abs(pca.components_[0]).round(3))))
    print(f'  Axe 2 ({v2*100:.1f}%) : ', dict(zip(cols_irs, abs(pca.components_[1]).round(3))))

## Étape 2 — Calculer l'IRS

In [ ]:
X_full = scaler_acp.transform(df[cols_irs].fillna(df[cols_irs].median()))
scores = pca.transform(X_full)

if n_axes == 1:
    df['IRS_brut'] = scores[:, 0]
else:
    v1 = pca.explained_variance_ratio_[0]
    v2 = pca.explained_variance_ratio_[1]
    total = v1 + v2
    df['IRS_brut'] = (v1/total) * scores[:, 0] + (v2/total) * scores[:, 1]

# Normaliser entre 0 et 1
irs_min = df['IRS_brut'].min()
irs_max = df['IRS_brut'].max()
df['IRS'] = (df['IRS_brut'] - irs_min) / (irs_max - irs_min)
print(f'IRS calculé — Moyenne : {df["IRS"].mean():.3f}')

## Étape 3 — Seuils par percentiles
**Pourquoi les percentiles ?** Défendables devant les juges.
CRITIQUE = pire que 90% des jours observés au Cameroun.

In [ ]:
# Seuils OMS AQG 2021 — NCBI NBK574591, Table 3.6
# PM2.5 24h : AQG=15, IT-4=25, IT-3=37.5, IT-2=50, IT-1=75
SEUIL_AQG  = 15
SEUIL_IT4  = 25
SEUIL_IT3  = 37.5
SEUIL_IT2  = 50
SEUIL_IT1  = 75

print('Seuils PM2.5 — OMS AQG 2021 (NCBI NBK574591, Table 3.6) :')
print(f'  🟢 FAIBLE      : PM2.5 < {SEUIL_AQG}     µg/m³  (AQG OMS)')
print(f'  🟡 MODÉRÉ      : PM2.5 < {SEUIL_IT4}     µg/m³  (IT-4)')
print(f'  🟠 ÉLEVÉ       : PM2.5 < {SEUIL_IT3}   µg/m³  (IT-3)')
print(f'  🔴 TRÈS ÉLEVÉ  : PM2.5 < {SEUIL_IT2}     µg/m³  (IT-2)')
print(f'  🟣 CRITIQUE    : PM2.5 < {SEUIL_IT1}     µg/m³  (IT-1)')
print(f'  ⚫ DANGEREUX   : PM2.5 ≥ {SEUIL_IT1}     µg/m³  (au-delà IT-1)')

# Statistiques descriptives
print('\nDistribution PM2.5 dans vos données :')
for niveau, seuil_bas, seuil_haut in [
    ('🟢 FAIBLE',     0,          SEUIL_AQG),
    ('🟡 MODÉRÉ',     SEUIL_AQG,  SEUIL_IT4),
    ('🟠 ÉLEVÉ',      SEUIL_IT4,  SEUIL_IT3),
    ('🔴 TRÈS ÉLEVÉ', SEUIL_IT3,  SEUIL_IT2),
    ('🟣 CRITIQUE',   SEUIL_IT2,  SEUIL_IT1),
    ('⚫ DANGEREUX',  SEUIL_IT1,  9999),
]:
    n = ((df['pm2_5_moyen'] >= seuil_bas) & 
         (df['pm2_5_moyen'] <  seuil_haut)).sum()
    pct = n / len(df) * 100
    print(f'  {niveau:20s} : {n:6,} jours ({pct:.1f}%)')

# Classification
df['niveau_sanitaire'] = df['pm2_5_moyen'].apply(
    lambda x: '🟢 FAIBLE'      if pd.isna(x) else
              '🟢 FAIBLE'      if x < SEUIL_AQG  else
              '🟡 MODÉRÉ'      if x < SEUIL_IT4  else
              '🟠 ÉLEVÉ'       if x < SEUIL_IT3  else
              '🔴 TRÈS ÉLEVÉ'  if x < SEUIL_IT2  else
              '🟣 CRITIQUE'    if x < SEUIL_IT1  else
              '⚫ DANGEREUX'
)

print(f'\n✅ niveau_sanitaire créé — basé sur OMS AQG 2021')
print(df['niveau_sanitaire'].value_counts())

## Étape 4 — Sauvegarder

In [ ]:
joblib.dump(scaler_acp, '../models/scaler_acp_irs.pkl')
joblib.dump(pca,        '../models/pca_irs.pkl')
joblib.dump(cols_irs,   '../models/cols_irs.pkl')

# Seuils OMS AQG 2021 — NCBI NBK574591, Table 3.6
joblib.dump({
    'AQG' : SEUIL_AQG,   # 15  µg/m³
    'IT4' : SEUIL_IT4,   # 25  µg/m³
    'IT3' : SEUIL_IT3,   # 37.5 µg/m³
    'IT2' : SEUIL_IT2,   # 50  µg/m³
    'IT1' : SEUIL_IT1,   # 75  µg/m³
    'irs_min' : irs_min,
    'irs_max' : irs_max
}, '../models/seuils_irs.pkl')

df.to_excel('../data/processed/dataset_final.xlsx', index=False)
print('✅ IRS calculé et sauvegardé')
print('✅ Seuils OMS AQG 2021 sauvegardés')
print('✅ Dataset final sauvegardé : data/processed/dataset_final.xlsx')

In [ ]:
# Sauvegarde Excel + Parquet
df.to_excel('../data/processed/dataset_final.xlsx', index=False)
df.to_parquet('../data/processed/dataset_final.parquet', index=False)
print('✅ Dataset final sauvegardé en Excel et Parquet')
